# 03 · XLM-R with controlled long-text selection

The first transformer reference uses `FacebookAI/xlm-roberta-base` for English
and German `human`, `ai` and `fusion` classification. Does selecting both document
ends improve development macro-F1 over taking only the beginning at an identical
512-token budget? Neither policy is assumed superior before measurement.

The comparison reuses the classical baseline's complete frozen holdout. Model
fitting and checkpoint selection use development data only. This notebook audits
the experiment and reviews saved measurements; GPU execution is explicit in
`experiments/02_xlm_roberta_kaggle.ipynb`. No model is trained by running this notebook.

In [1]:
import json
import sys
import tomllib
from pathlib import Path

import plotly.express as px
import plotly.io
import polars as pl
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from hm3.encoder_data import LABEL2ID, validate_encoder_manifest
from hm3.validation import LABELS, fingerprint_file
from hm3.xlmr_checkpoints import verify_checkpoint

CONFIG = tomllib.loads((ROOT / "config/xlmr.toml").read_text())
METADATA = ROOT / "data/metadata"
KAGGLE_OUTPUT_ROOT = ROOT / "runs/kaggle/notebookf475a1574b/hm3_xlmr"
pointer = json.loads((KAGGLE_OUTPUT_ROOT / "current_run.json").read_text())
RESULTS_DIRECTORY: Path | None = KAGGLE_OUTPUT_ROOT / pointer["run_directory"]
plotly.io.renderers.default = "plotly_mimetype"
CLASS_COLORS = {"human": "#2A9D8F", "ai": "#E76F51", "fusion": "#577590"}
display({"seed": CONFIG["seed"], "label_to_id": LABEL2ID, "results": RESULTS_DIRECTORY})

{'seed': 2026,
 'label_to_id': {'human': 0, 'ai': 1, 'fusion': 2},
 'results': PosixPath('/Users/nikitalenyk/Desktop/hm3/runs/kaggle/notebookf475a1574b/hm3_xlmr/runs/xlmr_head_only_40k/57dfda5d5e0e10df1bdc79aba5ff37c7d140c53dd8b08832e6d9eb6f370b7b2a')}

## 01 · Frozen validation and shared training IDs

**Question.** Can both policies use identical development populations without
changing the established holdout?

Confirmed normalized duplicate groups stay together. Within each joint
label/language/domain bucket, seeded whole-group sampling reserves approximately
2k development evaluation rows, then approximately 40k training rows. Rare buckets
prioritize training; missing evaluation strata and rounding deviations are recorded.
These files are private generated metadata and must be restored with their JSON
provenance. Existing incompatible files fail rather than regenerate silently.

In [2]:
frozen_path = METADATA / "validation_manifest.parquet"
selected_path = METADATA / "xlmr_development_40k.parquet"
if fingerprint_file(frozen_path) != CONFIG["frozen_manifest_sha256"]:
    raise ValueError("Restore the original frozen validation manifest")
if fingerprint_file(selected_path) != CONFIG["encoder_manifest_sha256"]:
    raise ValueError("Restore the common frozen encoder selection")
frozen = pl.read_parquet(frozen_path)
selected = pl.read_parquet(selected_path)
validate_encoder_manifest(frozen, selected)
sampling = json.loads(selected_path.with_suffix(".json").read_text())
if sampling["parent_manifest_sha256"] != CONFIG["frozen_manifest_sha256"]:
    raise ValueError("Sampling provenance does not match the frozen split")
display(selected.group_by("role").len().sort("role"))
display(pl.DataFrame(sampling["distribution_deviations"]))
display(
    {
        "missing_evaluation_strata": sampling["evaluation_strata_missing"],
        "confirmed_group_overlap": sampling["known_group_overlap"],
        "remaining_risk": sampling["unresolved_risk"],
    }
)

role,len
str,u32
"""development_eval""",2001
"""holdout""",54471
"""train""",40001
"""unused""",175879


role,max_joint_deviation_pp
str,f64
"""train""",0.001338
"""development_eval""",0.038501


{'missing_evaluation_strata': [],
 'confirmed_group_overlap': 0,
 'remaining_risk': 'Related rewrites without confirmed group IDs may cross roles'}

In [3]:
role_class_counts = selected.group_by("role", "label").len().sort("role", "label")
role_language_counts = selected.group_by("role", "lang").len().sort("role", "lang")
joint_counts = selected.group_by("role", "label", "lang", "domain").len()
display(role_class_counts.pivot(on="label", index="role", values="len"))
display(role_language_counts.pivot(on="lang", index="role", values="len"))
display(
    joint_counts.filter(pl.col("role").is_in(["train", "development_eval"])).sort("len").head(12)
)
role_domain_counts = selected.filter(pl.col("role").is_in(["train", "development_eval"]))
role_domain_counts = role_domain_counts.group_by("role", "domain").len()
display(role_domain_counts.pivot(on="role", index="domain", values="len").sort("domain"))

role,ai,fusion,human
str,u32,u32,u32
"""development_eval""",895,615,491
"""holdout""",24386,16738,13347
"""train""",17909,12291,9801
"""unused""",78750,54038,43091


role,de-DE,en-EN
str,u32,u32
"""development_eval""",389,1612
"""holdout""",10554,43917
"""train""",7751,32250
"""unused""",34077,141802


role,label,lang,domain,len
str,str,str,str,u32
"""development_eval""","""human""","""de-DE""","""gutenberg""",1
"""development_eval""","""fusion""","""de-DE""","""gutenberg""",1
"""development_eval""","""human""","""en-EN""","""gutenberg""",2
"""development_eval""","""fusion""","""en-EN""","""gutenberg""",2
"""development_eval""","""ai""","""de-DE""","""gutenberg""",3
…,…,…,…,…
"""train""","""fusion""","""de-DE""","""gutenberg""",6
"""development_eval""","""fusion""","""en-EN""","""arxiv_papers""",13
"""train""","""human""","""en-EN""","""gutenberg""",32


domain,train,development_eval
str,u32,u32
"""arxiv_papers""",2099,104
"""blog_authorship_corpus""",5955,298
"""bundestag""",7687,384
"""cnn_news""",6504,325
"""euro_court_cases""",4050,203
"""gutenberg""",1080,56
"""house_of_commons""",4679,234
"""student_essays""",7947,397


**Measured population audit.** Whole-group rounding gives 40,001 training rows,
2,001 development evaluation rows and the original 54,471 holdout rows. Maximum
joint-distribution deviations from the full development partition are 0.00134
and 0.03850 percentage points respectively. No evaluation joint stratum is missing.
Counts above provide the reproducible evidence; all policies reuse these exact IDs.

**Limitation.** Source-document family identifiers are unavailable. Equality groups
block known duplication, but related rewrites can still cross development and
holdout. This protocol does not establish that validation is leakage-free.

## 02 · Visible experiment matrix

EDA established substantial long-text truncation risk, motivating a controlled
token-selection comparison. Language and domain remain diagnostic metadata rather
than explicit predictive inputs. The model receives original text without
translation or destructive normalization.

The output mapping is **human=0, ai=1, fusion=2**. Checkpoint configuration,
probability columns and logits must use this order. Document-level `fusion`
labels are never treated as ground truth for individual chunks.

In [4]:
common = {key: value for key, value in CONFIG.items() if key != "experiments"}
experiment_matrix = pl.DataFrame(
    [{"experiment": name, **common, **settings} for name, settings in CONFIG["experiments"].items()]
)
display(
    experiment_matrix.select(
        "experiment",
        "truncation_policy",
        "model_id",
        "max_length",
        "num_train_epochs",
        "learning_rate",
        "weight_decay",
        "seed",
    )
)
display(
    {
        key: CONFIG[key]
        for key in [
            "revision",
            "per_device_train_batch_size",
            "per_device_eval_batch_size",
            "gradient_accumulation_steps",
            "fp16",
            "gradient_checkpointing",
            "eval_steps",
            "save_steps",
            "early_stopping_patience",
        ]
    }
)

experiment,truncation_policy,model_id,max_length,num_train_epochs,learning_rate,weight_decay,seed
str,str,str,i64,i64,f64,f64,i64
"""xlmr_head_only_40k""","""head""","""FacebookAI/xlm-roberta-base""",512,2,0.00002,0.01,2026
"""xlmr_head_tail_40k""","""head_tail""","""FacebookAI/xlm-roberta-base""",512,2,0.00002,0.01,2026


{'revision': 'e73636d4f797dec63c3081bb6ed5c7b0bb3f2089',
 'per_device_train_batch_size': 4,
 'per_device_eval_batch_size': 8,
 'gradient_accumulation_steps': 8,
 'fp16': True,
 'gradient_checkpointing': True,
 'eval_steps': 250,
 'save_steps': 250,
 'early_stopping_patience': 2}

## 03 · Token budget, optimizer and checkpoint selection

For one XLM-R sequence, `<s>` and `</s>` occupy two positions. The 512-token
budget therefore contains **510 content tokens**: head-only takes the first 510;
head-plus-tail takes the first 255 and last 255. Short documents retain all content.
Dynamic padding adds batch padding and an attention mask; it does not change
retained content or the sequence budget. These are subword tokens, not words.

Initial controls are AdamW, learning rate `2e-5`, weight decay `0.01`, at most two
epochs, seed 2026, FP16, gradient checkpointing, batch 4 and accumulation 8 on
one GPU (effective batch 32). Development evaluation occurs every 250 optimizer
steps. Two non-improving development evaluations trigger early stopping; the
highest development macro-F1 checkpoint is restored. The holdout is excluded
from training, early stopping and checkpoint selection.

The reusable call is `train_encoder(train_tokens, development_tokens, tokenizer,
settings, run_path, identity, cache_dir=cache_path)`. Each token table contains
stable IDs and document labels; the hosted entry point shows the selected
configuration and calls this only when `MODE = "train"`. The second policy
requires a separate explicit selection. No GPU training occurs here.

## 04 · Verified measured results

The default review restores the completed Kaggle head-only run. Prediction IDs and
checksums are verified before displaying metrics; selected-model file integrity is
checked without loading model weights. For a clone without downloaded outputs,
restore the named Kaggle output tree before rerunning this review.

The saved run used the original gradient-accumulation handling described below. Its
measurements remain unchanged. A full-data refit is a separate procedure and has no
valid local holdout score because its training population includes the holdout.

In [5]:
results = None
predictions = None
if RESULTS_DIRECTORY is not None:
    record = json.loads((RESULTS_DIRECTORY / "run_metadata.json").read_text())
    identity = record["identity"]
    saved_configuration = identity["configuration"]
    if identity["scope"] != "sampled" or identity["label_order"] != list(LABELS):
        raise ValueError("Research evaluation requires the sampled three-class experiment")
    for key in ["source_sha256", "frozen_manifest_sha256", "encoder_manifest_sha256"]:
        if saved_configuration[key] != CONFIG[key]:
            raise ValueError(f"Saved results use a different {key}")
    verify_checkpoint(RESULTS_DIRECTORY / "selected_model", identity, False)
    receipt = json.loads((RESULTS_DIRECTORY / "holdout_record.json").read_text())
    if receipt["identity"] != identity or receipt["model_sha256"] != fingerprint_file(
        RESULTS_DIRECTORY / "selected_model/checkpoint_manifest.json"
    ):
        raise ValueError("Holdout receipt does not match selected weights")
    for name, expected in receipt["files"].items():
        if fingerprint_file(RESULTS_DIRECTORY / name) != expected:
            raise ValueError(f"Saved result checksum mismatch: {name}")
    predictions = pl.read_parquet(RESULTS_DIRECTORY / "holdout_predictions.parquet")
    expected_ids = selected.filter(pl.col("role") == "holdout")["id"].sort()
    if predictions["id"].n_unique() != len(expected_ids) or not predictions["id"].sort().equals(
        expected_ids
    ):
        raise ValueError("Holdout predictions do not cover the original fixed IDs")
    expected_labels = selected.filter(pl.col("role") == "holdout").select("id", "label")
    if (
        predictions.join(expected_labels, on="id", suffix="_expected")
        .filter(pl.col("label") != pl.col("label_expected"))
        .height
    ):
        raise ValueError("Saved true labels differ from the frozen manifest")
    results = json.loads((RESULTS_DIRECTORY / "holdout_metrics.json").read_text())
    display(saved_configuration)
    display({key: results[key] for key in ["macro_f1", "weighted_f1", "accuracy"]})
    display(pl.DataFrame(results["per_class"]))
else:
    display(
        pl.DataFrame(
            {
                "experiment": list(CONFIG["experiments"]),
                "execution_status": ["not measured", "not measured"],
            }
        )
    )

{'model_id': 'FacebookAI/xlm-roberta-base',
 'revision': 'e73636d4f797dec63c3081bb6ed5c7b0bb3f2089',
 'seed': 2026,
 'train_target': 40000,
 'evaluation_target': 2000,
 'max_length': 512,
 'num_train_epochs': 2,
 'learning_rate': 2e-05,
 'weight_decay': 0.01,
 'per_device_train_batch_size': 4,
 'per_device_eval_batch_size': 8,
 'gradient_accumulation_steps': 8,
 'fp16': True,
 'gradient_checkpointing': True,
 'eval_steps': 250,
 'save_steps': 250,
 'logging_steps': 50,
 'early_stopping_patience': 2,
 'minimum_free_disk_gib': 12,
 'source_sha256': 'd55823815042eed2937e92e1018f5595915b8f7573d861048956521434f65025',
 'frozen_manifest_sha256': '1f12f2a2b312b513ace36966e9b3a011e7e3b279aeed6dd254173ac9a3d26a5d',
 'encoder_manifest_sha256': 'f37face1dcb866272bb7902e5c3bd8f2fd43b7d7e01424d048de50f1ca0a1ef5',
 'truncation_policy': 'head',
 'experiment': 'xlmr_head_only_40k'}

{'macro_f1': 0.8134274235569098,
 'weighted_f1': 0.8406405740985665,
 'accuracy': 0.8428154430798039}

label,precision,recall,f1,support
str,f64,f64,f64,i64
"""human""",0.675021,0.829325,0.74426,13347
"""ai""",0.964339,0.989133,0.976578,24386
"""fusion""",0.82075,0.640399,0.719444,16738


## 05 · Training stability and class confusion

**Question.** Does development loss/F1 indicate stable fitting, and which mistakes
dominate the selected model's holdout errors? Development metrics choose the
checkpoint; holdout metrics describe its generalization after that choice.
Accuracy and weighted-F1 complement the primary fixed three-class macro-F1.

In [6]:
if RESULTS_DIRECTORY is not None and results is not None:
    events = json.loads((RESULTS_DIRECTORY / "history.json").read_text())["events"]
    history = pl.from_dicts(events, infer_schema_length=None)
    loss_columns = [name for name in ["loss", "eval_loss"] if name in history.columns]
    if loss_columns:
        loss = (
            history.select("step", *loss_columns)
            .unpivot(index="step", variable_name="metric", value_name="logged_loss")
            .drop_nulls()
        )
        display(
            px.line(
                loss,
                x="step",
                y="logged_loss",
                color="metric",
                title="Historical logged losses (different normalization)",
            )
        )
    score_columns = [name for name in history.columns if name.startswith("eval_f1_")]
    if "eval_macro_f1" in history.columns:
        score_columns.insert(0, "eval_macro_f1")
    if score_columns:
        scores = (
            history.select("step", *score_columns)
            .unpivot(index="step", variable_name="metric", value_name="f1")
            .drop_nulls()
        )
        display(
            px.line(
                scores,
                x="step",
                y="f1",
                color="metric",
                title="Development-only checkpoint metrics",
            )
        )
    for key, title in [
        ("confusion_matrix", "Fixed-holdout confusion counts"),
        ("row_normalized_confusion_matrix", "Fixed-holdout confusion by true class"),
    ]:
        display(
            px.imshow(
                results[key],
                x=list(LABELS),
                y=list(LABELS),
                text_auto=True,
                title=title,
                labels={"x": "Predicted label", "y": "True label", "color": "Value"},
            )
        )

## 06 · Language, domain, length and fusion diagnostics

Subgroup macro-F1 always averages over human, ai and fusion, with undefined
class scores set to zero. Report class supports and missing-class flags; a small
or missing-class subgroup cannot be compared as if all three classes had strong
support. Length buckets are fixed word-count intervals, not tokenizer lengths.
Subgroup results are diagnostics and do not authorize holdout-driven tuning.

In [7]:
if predictions is not None:
    from hm3.evaluation import subgroup_metrics

    for columns in [["lang"], ["domain"], ["length_bucket"], ["lang", "domain"]]:
        display(subgroup_metrics(predictions, columns))
    fusion_errors = predictions.filter(
        (pl.col("label") != pl.col("prediction"))
        & ((pl.col("label") == "fusion") | (pl.col("prediction") == "fusion"))
    )
    display(fusion_errors.group_by("label", "prediction").len().sort("len", descending=True))
    display(
        predictions.filter(pl.col("label") != pl.col("prediction"))
        .sort("id")
        .sample(
            n=min(12, predictions.filter(pl.col("label") != pl.col("prediction")).height), seed=2026
        )
        .select("id", "label", "prediction", "lang", "domain", "words")
    )

lang,support,observed_classes,has_sufficient_support,macro_f1,human_support,ai_support,fusion_support
str,i64,i64,bool,f64,i64,i64,i64
"""de-DE""",10554,3,true,0.746439,2902,4067,3585
"""en-EN""",43917,3,true,0.828915,10445,20319,13153


domain,support,observed_classes,has_sufficient_support,macro_f1,human_support,ai_support,fusion_support
str,i64,i64,bool,f64,i64,i64,i64
"""arxiv_papers""",2859,3,true,0.620017,82,2410,367
"""blog_authorship_corpus""",8110,3,true,0.741162,2493,3137,2480
"""bundestag""",10469,3,true,0.746145,2896,3995,3578
"""cnn_news""",8857,3,true,0.845724,2679,3898,2280
"""euro_court_cases""",5516,3,true,0.805571,1256,2228,2032
"""gutenberg""",1468,3,true,0.670181,49,1343,76
"""house_of_commons""",6372,3,true,0.825514,1391,3059,1922
"""student_essays""",10820,3,true,0.890513,2501,4316,4003


length_bucket,support,observed_classes,has_sufficient_support,macro_f1,human_support,ai_support,fusion_support
str,i64,i64,bool,f64,i64,i64,i64
"""001-128""",5520,3,true,0.740461,1517,2461,1542
"""1025+""",4698,3,true,0.709682,1597,1107,1994
"""129-512""",25240,3,true,0.86576,5596,11488,8156
"""513-1024""",19013,3,true,0.790323,4637,9330,5046


lang,domain,support,observed_classes,has_sufficient_support,macro_f1,human_support,ai_support,fusion_support
str,str,i64,i64,bool,f64,i64,i64,i64
"""de-DE""","""bundestag""",10469,3,true,0.746145,2896,3995,3578
"""de-DE""","""gutenberg""",85,3,false,0.762821,6,72,7
"""en-EN""","""arxiv_papers""",2859,3,true,0.620017,82,2410,367
"""en-EN""","""blog_authorship_corpus""",8110,3,true,0.741162,2493,3137,2480
"""en-EN""","""cnn_news""",8857,3,true,0.845724,2679,3898,2280
"""en-EN""","""euro_court_cases""",5516,3,true,0.805571,1256,2228,2032
"""en-EN""","""gutenberg""",1383,3,true,0.659635,43,1271,69
"""en-EN""","""house_of_commons""",6372,3,true,0.825514,1391,3059,1922
"""en-EN""","""student_essays""",10820,3,true,0.890513,2501,4316,4003


label,prediction,len
str,str,u32
"""fusion""","""human""",5256
"""human""","""fusion""",2149
"""fusion""","""ai""",763
"""ai""","""fusion""",192


id,label,prediction,lang,domain,words
str,str,str,str,str,u32
"""6c53f2""","""human""","""fusion""","""en-EN""","""house_of_commons""",300
"""9d0b61""","""human""","""fusion""","""en-EN""","""euro_court_cases""",1677
"""b7e6ef""","""human""","""fusion""","""de-DE""","""bundestag""",590
"""ddc0a4""","""human""","""fusion""","""en-EN""","""cnn_news""",526
"""076be4""","""human""","""fusion""","""en-EN""","""house_of_commons""",165
…,…,…,…,…,…
"""e6f45a""","""fusion""","""human""","""en-EN""","""blog_authorship_corpus""",412
"""d3c23a""","""fusion""","""human""","""en-EN""","""cnn_news""",593
"""78f9f6""","""human""","""fusion""","""en-EN""","""blog_authorship_corpus""",156


In [8]:
comparison_path = ROOT / "data/metadata/model_comparison.json"
comparison_record = json.loads(comparison_path.read_text())
if comparison_record["frozen_manifest_sha256"] != CONFIG["frozen_manifest_sha256"]:
    raise ValueError("Comparison uses a different frozen validation split")
for model in comparison_record["models"].values():
    if (
        fingerprint_file(ROOT / model["run_directory"] / "holdout_record.json")
        != model["holdout_receipt_sha256"]
    ):
        raise ValueError("Comparison receipt changed; recompute measurements")
display(
    pl.DataFrame(
        [
            {
                "model": (
                    "Word + character TF-IDF + LR"
                    if name == "baseline"
                    else "XLM-R head-only (40k; original accumulation)"
                ),
                "training_population": 217881 if name == "baseline" else 40001,
                "holdout_macro_f1": model["metrics"]["macro_f1"],
                "human_f1": model["metrics"]["per_class"][0]["f1"],
                "ai_f1": model["metrics"]["per_class"][1]["f1"],
                "fusion_f1": model["metrics"]["per_class"][2]["f1"],
            }
            for name, model in comparison_record["models"].items()
        ]
    )
)

model,training_population,holdout_macro_f1,human_f1,ai_f1,fusion_f1
str,i64,f64,f64,f64,f64
"""Word + character TF-IDF + LR""",217881,0.89572,0.841083,0.989446,0.856631
"""XLM-R head-only (40k; original…",40001,0.813427,0.74426,0.976578,0.719444


## 07 · Measured comparison and limitations

The verified common holdout contains 54,471 documents. Word + character TF-IDF with
LR reaches **0.895720** macro-F1; this XLM-R head-only run reaches **0.813427**, a
0.082292 difference. Training populations differ: 217,881 development texts for the
final baseline versus a 40,001-document encoder sample with a separate 2,001-document
development evaluation set. This does not establish an architecture ranking under
matched data or compute.

XLM-R fusion F1 is **0.719444**, compared with 0.856631 for the baseline. Of 16,738
fusion documents, XLM-R predicts 5,256 as human (31.40%) and 763 as AI (4.56%).
The baseline makes 2,631 and 251 such errors. XLM-R macro-F1 is 0.746439 on German
and 0.828915 on English; different domain mixtures confound a language-only explanation.
For documents over 1,024 words, XLM-R reaches 0.709682 versus baseline 0.912023.
Truncation is a plausible contributor, but this subgroup association does not prove causation.

**Development checkpoint selection.** The best development macro-F1 is **0.819411**
at step 500, epoch 0.39996. Later development macro-F1 is 0.817034 at step 750 and
0.728991 at step 1,000. Patience two stops fitting at epoch 0.79992, before the initial
two-epoch ceiling. Trainer reports 1,446.568 seconds of training, about 24.11 minutes;
this excludes setup and the separate holdout stage. The selected step-500 weights,
not the terminal step-1,000 weights, generated holdout predictions.

**Confirmed implementation issue.** With Transformers 5.24, XLM-R's `forward(**kwargs)`
causes Trainer to infer that the model normalizes loss across accumulated batches.
The classification head actually returns mean document cross-entropy. A tiny CPU
reproduction with accumulation 8 returned loss 1.098394 where 0.137299 was expected,
an eightfold difference; declaring `accepts_loss_kwargs=False` restores correct
normalization. This affects backpropagation and training-loss scale. The current
loader fixes future runs; the stored run, checkpoints and scores are preserved.
The exact effect on macro-F1 is unmeasured, so the observed score gap cannot be
attributed to this issue alone. Training and evaluation loss curves from this
historical run must not be compared as if they shared the same loss normalization.

A corrected head-only run should precede a controlled head-plus-tail comparison,
using the same frozen IDs and development-only checkpoint rule. Do not resume the
historical training state with changed loss semantics or tune using holdout scores.
Head-plus-tail remains unmeasured. Sliding windows and document aggregation remain
later experiments; document labels do not identify authorship of individual chunks.
Unknown source-family leakage, one random seed and repeated-holdout selection risks remain.